# Integrating sbi-trained likelihoods into HSSM (NRE via ONNX)

This tutorial mirrors the structure of `bayesflow_lre_integration.ipynb` for the
third major SBI library in the HSSM ecosystem: **[sbi](https://github.com/sbi-dev/sbi)**
(mackelab). It demonstrates how to:

1. Train a neural ratio estimator (NRE) on synthetic DDM simulations using sbi.
2. Export the trained estimator to ONNX via
   [`lanfactory.onnx.transform_sbi_to_onnx`](https://alexanderfengler.github.io/LANfactory/exporting_sbi_models/).
3. Load the ONNX file into HSSM exactly like any other LAN-style approximator and run
   MCMC inference.
4. Compare against HSSM's analytical DDM likelihood as the gold-standard posterior.

> **Why NRE only, not NLE/MNLE?**
> Vanilla NLE with a MAF flow misbehaves on DDM data because rt is continuous but
> choice is discrete (∈ {−1, +1}). The flow treats choice as continuous, can't
> represent the support boundary `rt > t_nd`, and produces qualitatively wrong
> posteriors (we observed v ≈ 0.12 vs truth 0.5, with spurious bimodality on a).
> The correct sbi method is **MNLE** (Mixed Neural Likelihood Estimator), which
> splits x into discrete and continuous dims and models each properly. But MNLE's
> categorical lookup uses `torch.searchsorted`, which `torch.onnx.export` doesn't
> support and `jaxonnxruntime` lacks a handler for. See
> [plans/sbi-onnx-integration.md](../../../HSSMSpine/plans/sbi-onnx-integration.md)
> "Deferred sbi paths" for the resolution roadmap (a ~50-line upstream PR to
> `jaxonnxruntime` unlocks both MNLE and NSF flows in one stroke).
>
> Until then, **NRE is the working integration path** — it doesn't need to model
> density at all, just a classifier between joint and marginal pairs, which is
> robust to the discrete/continuous mixing.

> **Environment note:** This tutorial requires both `hssm` and `lanfactory[all]` (which
> pulls `sbi` and `nflows`) in the same environment. JAX/flax/numpyro pins must be
> resolved jointly across the two packages.

## Part 1 — Setup

In [ ]:
# Enable x64 BEFORE any other JAX-touching import.
# HSSM's onnx2jax auto-flips this when loading sbi-exported ONNX graphs that carry
# int64 tensors (typical for normalizing flows from torch.onnx.export), but setting
# it explicitly here is best practice and silences the auto-flip UserWarning.
import jax
jax.config.update("jax_enable_x64", True)

from pathlib import Path
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

import hssm
from sbi.inference import NRE_A
from sbi.neural_nets import classifier_nn
from sbi.utils import BoxUniform
from ssms.basic_simulators.simulator import simulator

# Import transform_sbi_to_onnx with a fallback. The clean path works when
# lanfactory is installed in an env whose JAX/flax pins are compatible with this
# notebook's other deps. Until the cross-repo env alignment lands, the fallback
# loads only the sbi exporter module directly, bypassing lanfactory's top-level
# __init__.py (which would otherwise pull the flax-dependent jax_mlp trainer).
try:
    from lanfactory.onnx import transform_sbi_to_onnx
except ImportError:
    import importlib.util
    import os

    # Try candidates: explicit env var, then several relative paths covering
    # common Jupyter launch contexts (notebook dir / repo root / spine root).
    _candidates = [
        os.environ.get("LANFACTORY_SBI_PATH"),
        "../../../LANfactory/src/lanfactory/onnx/sbi.py",  # cwd = notebook dir
        "repos/LANfactory/src/lanfactory/onnx/sbi.py",     # cwd = HSSMSpine root
        "../LANfactory/src/lanfactory/onnx/sbi.py",        # cwd = repos/HSSM root
    ]
    _path = None
    for _c in _candidates:
        if _c and Path(_c).exists():
            _path = Path(_c).resolve()
            break
    if _path is None:
        raise ImportError(
            "Could not locate lanfactory/onnx/sbi.py. Set the LANFACTORY_SBI_PATH "
            "environment variable to the absolute path of that file, or run the "
            "notebook from a directory where one of the relative candidates resolves: "
            f"{[c for c in _candidates if c]}"
        )
    _spec = importlib.util.spec_from_file_location("_lanfactory_sbi", _path)
    _mod = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    transform_sbi_to_onnx = _mod.transform_sbi_to_onnx
    print(f"(fallback) loaded transform_sbi_to_onnx from {_path}")

np.random.seed(0)
torch.manual_seed(0)

# Training budget for NRE_A — the last known-working configuration.
# 1M (theta, x) pairs from ssm-simulators (1 sample per theta) at the wider
# HSSM-default prior bounds, with a moderate-sized MLP classifier.
# We are deliberately reverting to a simple, previously-validated setup to
# rule out new variables (NRE_B / multi-sample-per-theta / FCEmbedding /
# bigger classifier) before introducing them again one at a time.
N_TRAIN = 1_000_000
N_OBS = 500
NUM_EPOCHS = 300
STOP_AFTER_EPOCHS = 50
TRAINING_BATCH_SIZE = 500
HIDDEN_FEATURES = 100
# Smaller MCMC budget for verification runs. Together with target_accept=0.8
# and max_tree_depth=8 below, this keeps the MCMC step bounded so we can
# diagnose training-quality issues without paying 30+ minutes per pass.
MCMC_DRAWS = 500
MCMC_TUNE = 500
MCMC_CHAINS = 2

## Part 2 — Simulate observed DDM data

We use the standard 4-parameter DDM (`v`, `a`, `z`, `t`) from `ssm-simulators`. The
true parameters and parameter ranges below match the BayesFlow LRE tutorial so that
posteriors from the two tutorials can be compared apples-to-apples.

In [ ]:
DDM_PARAM_NAMES = ["v", "a", "z", "t"]
# Training prior matches HSSM's default bounds for model="ddm" with
# loglik_kind="approx_differentiable" — see hssm.defaults.default_model_config.
# This is important: if the training prior is narrower than HSSM's posterior
# can explore, MCMC will walk into parameter regions the surrogate never saw
# and extrapolate badly.
PRIOR_LOW = np.array([-3.0, 0.3, 0.0, 0.0], dtype=np.float32)
PRIOR_HIGH = np.array([3.0, 2.5, 1.0, 2.0], dtype=np.float32)
TRUE_THETA = np.array([0.5, 1.2, 0.5, 0.25], dtype=np.float32)

out = simulator(theta=TRUE_THETA[None, :], model="ddm", n_samples=N_OBS)
obs_data = pd.DataFrame(
    {
        "rt": out["rts"].squeeze().astype(np.float32),
        "response": out["choices"].squeeze().astype(np.float32),
    }
)
print(f"observed: {len(obs_data)} trials at true theta = {TRUE_THETA}")
obs_data.head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for resp, color in zip([-1, 1], ["C0", "C1"]):
    mask = obs_data["response"] == resp
    ax.hist(
        obs_data.loc[mask, "rt"],
        bins=40,
        alpha=0.6,
        label=f"choice={int(resp)}",
        color=color,
    )
ax.set_xlabel("RT (s)")
ax.set_ylabel("count")
ax.set_title("Observed RT histogram by choice")
ax.legend()
plt.tight_layout()
plt.show()

## Part 3 — Train an sbi NRE_A classifier on DDM simulations

`NRE_A` (Hermans et al. 2020) learns a binary classifier that distinguishes
joint `(θ, x)` pairs from marginal `(θ', x)` pairs (where θ' is drawn from the
prior). The output logit equals `log p(x | θ) − log p(x)` up to a constant, so
it serves directly as the HSSM log-likelihood for MCMC (the θ-independent
constant drops out under MCMC's accept ratios).

This iteration uses the **last known-working NRE configuration**:

- `NRE_A` (binary classifier, not contrastive)
- 1M `(θ, x)` training pairs (1 sample per θ)
- `hidden_features = 100` (sbi default is 50)
- No embedding net on θ
- `norm_layer = nn.Identity` (jaxonnxruntime doesn't implement
  `LayerNormalization`, so the MLP norm layer is disabled)

We are reverting to this baseline because a more ambitious configuration
(NRE_B + atomic contrastive + multi-sample-per-θ + FCEmbedding +
`hidden_features=128`) produced a classifier that gave HSSM a near-constant
log-likelihood at MCMC time — the chains explored the entire prior with no
concentration. The bisect strategy is to verify this simpler config works,
then re-introduce changes one at a time.

In [ ]:
prior = BoxUniform(
    low=torch.from_numpy(PRIOR_LOW),
    high=torch.from_numpy(PRIOR_HIGH),
)
theta_train = prior.sample((N_TRAIN,))

# Batched ssm-simulators: theta of shape (N, 4) with n_samples=1 returns
# rts/choices of shape (N, 1). Much faster than a Python loop for large N.
sim = simulator(
    theta=theta_train.numpy().astype(np.float32),
    model="ddm",
    n_samples=1,
)
x_train = torch.from_numpy(
    np.stack(
        [sim["rts"].squeeze(-1), sim["choices"].squeeze(-1)], axis=-1
    ).astype(np.float32)
)
print(f"training set: theta={theta_train.shape}, x={x_train.shape}")

In [ ]:
# Build the classifier. LayerNorm is disabled because jaxonnxruntime
# doesn't implement LayerNormalization. No embedding net on theta in this
# baseline-revert iteration (see Part 3 markdown for the bisect context).
classifier_builder = classifier_nn(
    model="mlp",
    norm_layer=nn.Identity,
    hidden_features=HIDDEN_FEATURES,
)
inference_nre = NRE_A(prior=prior, classifier=classifier_builder)
classifier_nre = inference_nre.append_simulations(theta_train, x_train).train(
    training_batch_size=TRAINING_BATCH_SIZE,
    max_num_epochs=NUM_EPOCHS,
    stop_after_epochs=STOP_AFTER_EPOCHS,
)
classifier_nre.eval()
print("NRE_A training complete")

## Part 4 — Export the trained NRE to ONNX

The exporter wraps the classifier's `forward(theta, x)` logit as the HSSM
log-likelihood. No Jacobian correction is needed — ratios are invariant to the
z-score standardization sbi applies internally.

In [ ]:
onnx_dir = Path("./sbi_onnx_artifacts")
onnx_dir.mkdir(exist_ok=True)
nre_onnx_path = onnx_dir / "ddm_nre.onnx"

transform_sbi_to_onnx(
    classifier_nre,
    str(nre_onnx_path),
    mode="nre",
    example_theta_dim=4,
    example_x_dim=2,
)
print(f"exported NRE: {nre_onnx_path}  ({nre_onnx_path.stat().st_size:,} bytes)")

## Part 4b — Pre-MCMC verification: is the trained classifier any good?

Before paying the multi-minute MCMC cost, sanity-check two things:

1. **Logit sweep across θ-space.** Hold three θ dimensions at their true values
   and sweep the fourth across its prior range, plotting the summed classifier
   log-ratio on the observed data. A well-trained NRE shows a sharp peak near
   the true value with tens-to-hundreds of log units of vertical range. A
   flat curve (< 5 log units of range) is the smoking gun for "classifier
   collapsed" — MCMC will produce a posterior equal to the prior, no point
   continuing.

2. **ONNX export round-trip.** Compare `classifier_nre(theta, x).item()` to the
   exported ONNX file evaluated through `onnxruntime` on the same input. If
   they disagree, the export has introduced a bug and MCMC won't be running
   the model you think it is.

These cells use the in-memory `classifier_nre` from Part 3 and the exported
file from Part 4 — they're cheap, no MCMC required.

In [ ]:
# Diagnostic: how much does the NRE logit change as we sweep each theta dim?
sweep_pts = 50
sweep_results = {}
obs_x_t = torch.from_numpy(
    obs_data[["rt", "response"]].values.astype(np.float32)
)
theta_center = torch.tensor(TRUE_THETA, dtype=torch.float32)
for dim, name in enumerate(DDM_PARAM_NAMES):
    sweep = torch.linspace(PRIOR_LOW[dim], PRIOR_HIGH[dim], sweep_pts)
    logits = []
    for v in sweep:
        theta = theta_center.clone()
        theta[dim] = v
        theta_row = theta.unsqueeze(0).repeat(len(obs_x_t), 1)
        with torch.no_grad():
            logits.append(classifier_nre(theta_row, obs_x_t).sum().item())
    sweep_results[name] = (sweep.numpy(), np.array(logits))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, name in zip(axes, DDM_PARAM_NAMES):
    th, lp = sweep_results[name]
    ax.plot(th, lp - lp.max(), "C0-", linewidth=2)
    ax.axvline(TRUE_THETA[DDM_PARAM_NAMES.index(name)], color="red",
               linestyle="--", linewidth=2, label="true θ")
    ax.set_xlabel(name)
    ax.set_ylabel("Δ summed log-ratio (= 0 at max)")
    ax.set_title(f"sweep over {name}\n(vertical range = {np.ptp(lp):.2f})")
    ax.legend(fontsize=8)
fig.suptitle(
    "Trained NRE classifier: log-ratio along each θ axis (others held at truth)",
    y=1.02,
)
plt.tight_layout()
plt.show()

# Sanity print: vertical range per dim. < ~5 log units is bad.
print("\nPer-dim vertical range (max log-ratio − min log-ratio):")
for name in DDM_PARAM_NAMES:
    _, lp = sweep_results[name]
    print(f"  {name}: {np.ptp(lp):.2f}")
print("\nInterpretation:")
print("  > 50 log units : strong discriminative signal, classifier well-trained.")
print("  10–50          : moderate signal; should give a usable posterior.")
print("  < 5            : near-flat; classifier collapsed to uninformative.")

In [ ]:
# Diagnostic: does the exported ONNX match the torch classifier?
import onnxruntime as _ort

_sess = _ort.InferenceSession(str(nre_onnx_path))
_input_name = _sess.get_inputs()[0].name
_test_theta = torch.tensor([[0.5, 1.2, 0.5, 0.25]], dtype=torch.float32)
_test_x = torch.tensor([[0.5, 1.0]], dtype=torch.float32)
_combined = (
    torch.cat([_test_theta, _test_x], dim=-1).squeeze(0).numpy().astype(np.float32)
)

with torch.no_grad():
    _y_torch = float(classifier_nre(_test_theta, _test_x).item())
_y_ort = float(_sess.run(None, {_input_name: _combined})[0])

print(f"torch logit at (θ=true, x=(0.5, +1)):  {_y_torch:+.5f}")
print(f"ORT logit at (θ=true, x=(0.5, +1)):    {_y_ort:+.5f}")
print(f"|Δ|: {abs(_y_torch - _y_ort):.2e}")
print()
if abs(_y_torch - _y_ort) < 1e-4:
    print("→ Export is faithful; any pathology is in the trained classifier itself.")
else:
    print("→ Export disagrees with torch by more than 1e-4 — the exporter is dropping"
          " information on the way through ONNX, regardless of training quality.")

## Part 5 — High-level integration via `hssm.HSSM()`

HSSM's `loglik_kind="approx_differentiable"` path consumes the `.onnx` file
identically to a LAN-trained network. With `model="ddm"` HSSM already knows the
parameter list and response columns; we just hand it the file.

In [ ]:
model_nre = hssm.HSSM(
    data=obs_data,
    model="ddm",
    loglik_kind="approx_differentiable",
    loglik=str(nre_onnx_path),
    p_outlier=0,
)
print(model_nre)

In [ ]:
# Verification-budget MCMC. target_accept=0.8 (was 0.9) and max_tree_depth=8
# (caps NUTS at 256 leapfrog steps per draw instead of 1024) bound the per-step
# cost so a pathological surrogate geometry can't produce a multi-hour run.
# progressbar=True lets you actually see chain progress as it goes.
idata_nre = model_nre.sample(
    sampler="numpyro",
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    chains=MCMC_CHAINS,
    target_accept=0.8,
    progressbar=True,
    nuts_sampler_kwargs={"max_tree_depth": 8},
)

In [ ]:
summary_nre = az.summary(idata_nre, var_names=DDM_PARAM_NAMES)
summary_nre

In [ ]:
az.plot_trace(idata_nre, var_names=DDM_PARAM_NAMES)
plt.tight_layout()
plt.show()

### Part 5b — Diagnostic: is the NRE classifier itself biased, or is HSSM not finding its mode?

For NRE the logit `forward(theta, x)` equals `log p(x | θ) − log p(x)` up to a
constant. The θ-independent term cancels when we compare two θ values on the
same data, so we can use the *summed logit over trials* as a proxy for "which θ
does the classifier think makes the data more likely." If the classifier itself
prefers a θ far from the truth, training quality is the issue; if it prefers the
truth, MCMC is exploring elsewhere.

In [ ]:
posterior_mean_nre = {
    p: float(idata_nre.posterior[p].mean()) for p in DDM_PARAM_NAMES
}
obs_x_t = torch.from_numpy(
    obs_data[["rt", "response"]].values.astype(np.float32)
)

def total_logit(classifier, theta_dict):
    """Sum log-ratio logit over all observed trials at a single theta."""
    theta_row = torch.tensor(
        [[theta_dict[p] for p in DDM_PARAM_NAMES]], dtype=torch.float32
    ).repeat(len(obs_x_t), 1)
    with torch.no_grad():
        return classifier(theta_row, obs_x_t).sum().item()

lt_true = total_logit(classifier_nre, dict(zip(DDM_PARAM_NAMES, TRUE_THETA)))
lt_mean = total_logit(classifier_nre, posterior_mean_nre)

print(f"NRE total logit at true theta:      {lt_true:+.2f}")
print(f"NRE total logit at posterior mean:  {lt_mean:+.2f}")
print(f"Δ (mean − true):                    {lt_mean - lt_true:+.2f}")
print()
if lt_mean > lt_true + 5.0:
    print("→ NRE itself prefers the wrong θ by a large margin.")
    print("  Diagnosis: TRAINING QUALITY.")
elif lt_mean > lt_true:
    print("→ NRE mildly prefers the posterior mean over the truth.")
    print("  Could be marginal-vs-joint, mild miscalibration, or sampling.")
else:
    print("→ NRE prefers the truth; the wrong posterior mean is a sampling")
    print("  artifact (priors / init / mixing), not a training issue.")
print()
print(f"Posterior mean: {posterior_mean_nre}")
print(f"True theta:     {dict(zip(DDM_PARAM_NAMES, TRUE_THETA.tolist()))}")

## Part 6 — Ground-truth posterior via HSSM's analytical DDM

HSSM ships a closed-form analytical likelihood for the standard DDM
(`loglik_kind="analytical"`, the [Navarro & Fuss](https://psyarxiv.com/cwsbm/)
form). Running MCMC against this likelihood on the *same observed data*
produces what we should treat as the gold-standard posterior for this
model + dataset: any deviation of the sbi-NRE marginals from these is
*approximation error* in the neural surrogate, not intrinsic posterior width.

The analytical likelihood uses slightly different parameter bounds
(a, t unbounded above; otherwise the same) but on the observed data the
posterior concentrates regardless of the wider bound.

In [ ]:
model_analytical = hssm.HSSM(
    data=obs_data,
    model="ddm",
    loglik_kind="analytical",
    p_outlier=0,
)
idata_analytical = model_analytical.sample(
    sampler="numpyro",
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    chains=MCMC_CHAINS,
    target_accept=0.9,
    progressbar=False,
)
summary_analytical = az.summary(idata_analytical, var_names=DDM_PARAM_NAMES)
summary_analytical

## Part 7 — Posterior comparison: analytical vs sbi NRE

The keystone comparison. The analytical posterior is the gold standard for this
model + data; the sbi-NRE marginals are the approximation we built. Distance
between NRE and analytical is the surrogate's approximation error; distance
between the analytical posterior and the true θ is intrinsic posterior width
(the data simply isn't infinite at N=500 trials).

For the broader cross-tutorial comparison alongside the LAN baseline and
BayesFlow-LRE result on the same simulated data, load their cached posteriors
here and add panels to the plot.

> Reuses the exact `TRUE_THETA`, `N_OBS`, and prior ranges from
> `bayesflow_lre_integration.ipynb`, so the BayesFlow-LRE posteriors are
> directly comparable when added.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, name, true_val in zip(axes, DDM_PARAM_NAMES, TRUE_THETA):
    samples_ana = idata_analytical.posterior[name].values.flatten()
    samples_nre = idata_nre.posterior[name].values.flatten()
    ax.hist(
        samples_ana,
        bins=30,
        alpha=0.5,
        label="analytical (truth)",
        color="C2",
        density=True,
    )
    ax.hist(samples_nre, bins=30, alpha=0.5, label="sbi NRE", color="C1", density=True)
    ax.axvline(true_val, color="red", linestyle="--", linewidth=2, label="true θ")
    ax.set_xlabel(name)
    ax.set_title(f"posterior over {name}")
    ax.legend(fontsize=8)
fig.suptitle(
    "DDM posterior recovery: analytical (gold) vs sbi NRE", y=1.02
)
plt.tight_layout()
plt.show()

## Summary and deferred work

We trained an sbi NRE_B classifier on synthetic DDM data with FCEmbedding on
the parameters and atomic contrastive estimation (`num_atoms=20`), exported it
to ONNX via `lanfactory.onnx.transform_sbi_to_onnx`, and ran MCMC through
HSSM's existing `loglik_kind="approx_differentiable"` pipeline. The resulting
posterior is compared against HSSM's analytical DDM posterior on the same
data — that's our gold-standard reference.

**Why not NLE in this tutorial?**

We originally planned an NLE section too. Vanilla NLE with a MAF flow produces
qualitatively wrong posteriors on DDM data because rt is continuous but choice
is discrete (∈ {−1, +1}); the flow can't represent that structure or the hard
support boundary `rt > t_nd`. The correct sbi method is **MNLE** (Mixed Neural
Likelihood Estimator), which factorizes `p(rt, choice | θ) = p(choice | θ) ·
p(rt | choice, θ)`. But MNLE's `CategoricalMassEstimator` uses
`torch.searchsorted` for value-to-index lookup, which `torch.onnx.export` does
not support — blocking the ONNX export path until a `SearchSorted` ONNX-op
handler is added to `jaxonnxruntime`. The same gap blocks Neural Spline Flows.

See `plans/sbi-onnx-integration.md` "Deferred sbi paths (MNLE, vanilla NLE on
DDM, NSF flows)" in HSSMSpine for the resolution roadmap. A single ~50-line
upstream PR to `jaxonnxruntime` adding `SearchSorted` unlocks both NSF flows
AND MNLE in one stroke.

**Where to look next**

- LANfactory's [Exporting sbi Models guide](https://alexanderfengler.github.io/LANfactory/exporting_sbi_models/) — supported-architecture matrix, known constraints, troubleshooting.
- The BayesFlow LRE tutorial (`bayesflow_lre_integration.ipynb`) — the same DDM with a different SBI library, for cross-toolkit comparison.
- LAN tutorials (`main_tutorial.ipynb`) — the original LANfactory workflow this integration builds on top of.